#Prepare Your Dataset


In [21]:
import pandas as pd
from datasets import Dataset

# Load CSV
df = pd.read_csv("dataset_finetune_s.csv")

# Convert to Hugging Face Dataset
data = Dataset.from_pandas(df)

In [22]:
dataset = data.train_test_split(test_size=0.2)  
train_dataset = dataset['train']
test_dataset = dataset['test']

#Load Teacher and Student Models

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

teacher_model_name = "google/flan-t5-large"
student_model_name = "final_finetuned-base"

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)
teacher_model = AutoModelForSeq2SeqLM.from_pretrained(teacher_model_name)

student_tokenizer = AutoTokenizer.from_pretrained(student_model_name)
student_model = AutoModelForSeq2SeqLM.from_pretrained(student_model_name)

#Define Preprocessing Function

In [33]:
def preprocess_function_with_context(examples):
    inputs = [f"Context: {ctx} Question: {q}" for ctx, q in zip(examples["Context"], examples["Question"])]
    targets = examples["Answer"]

    teacher_inputs = teacher_tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    student_inputs = student_tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = student_tokenizer(targets, max_length=512, truncation=True, padding="max_length")["input_ids"]

    return {
        "input_ids": teacher_inputs["input_ids"],  # For both teacher and student
        "attention_mask": teacher_inputs["attention_mask"],  # For both teacher and student
        "labels": labels,  # Ground truth labels
    }

train_dataset = train_dataset.map(preprocess_function_with_context, batched=True)
test_dataset = test_dataset.map(preprocess_function_with_context, batched=True)


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [35]:
columns_to_keep = ["input_ids", "attention_mask", "labels"]  # Only keep these
train_dataset = train_dataset.remove_columns([col for col in train_dataset.column_names if col not in columns_to_keep])
test_dataset = test_dataset.remove_columns([col for col in test_dataset.column_names if col not in columns_to_keep])
print(train_dataset[0].keys())  # Should include only 'input_ids', 'attention_mask', 'labels'


dict_keys(['labels', 'input_ids', 'attention_mask'])


#Generate Soft Targets Using the Teacher Model

In [42]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
import torch
import numpy as np

# Use a data collator for padding and batching
data_collator = DataCollatorForSeq2Seq(tokenizer=teacher_tokenizer, model=teacher_model)

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=data_collator)
val_dataloader = DataLoader(test_dataset, batch_size=8, collate_fn=data_collator)

# Test a batch
batch = next(iter(train_dataloader))
print(batch.keys())  # Should output: dict_keys(['input_ids', 'attention_mask', 'labels'])

def generate_teacher_logits(dataset, model, batch_size=8):
    model.eval()
    dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    logits_list = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            decoder_input_ids = input_ids  # Optionally modify for different tasks

            # Forward pass through the model, getting logits from the teacher
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, decoder_input_ids=decoder_input_ids)
            logits = outputs.logits.cpu()

            # Save logits incrementally (to a file, or process here before appending)
            logits_list.append(logits.numpy())  # Append numpy array instead of the tensor

    return np.concatenate(logits_list, axis=0)

# Generate and add soft targets to the dataset
train_logits = generate_teacher_logits(train_dataset, teacher_model)
# If necessary, convert the logits to a list incrementally to avoid memory issues
train_dataset = train_dataset.add_column("teacher_logits", train_logits.tolist())


dict_keys(['input_ids', 'attention_mask', 'labels', 'decoder_input_ids'])


MemoryError: 

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from torch.utils.data import DataLoader, Subset
from datasets import load_dataset


# ---------------------------
# 1. Define Custom Student Model (FLAN-T5 Base)
# ---------------------------
class SmallT5(nn.Module):
    def __init__(self, model_name='google/flan-t5-base'):
        super(SmallT5, self).__init__()
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask=None):
        output = self.model(input_ids, attention_mask=attention_mask)
        return output.logits


# ---------------------------
# 2. Main Training Code
# ---------------------------
def main():

    # -----------------------------
    # Data Preparation (Using SQUAD for simplicity)
    # -----------------------------
    dataset = load_dataset("squad_v2", split='train[:1%]')
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

    def preprocess_function(examples):
        return tokenizer(examples['question'], examples['context'], truncation=True, padding="max_length", max_length=512)

    encoded_dataset = dataset.map(preprocess_function, batched=True)
    
    train_loader = DataLoader(encoded_dataset, batch_size=4, shuffle=True)

    # -----------------------------
    # Teacher Model (FLAN-T5 Large)
    # -----------------------------
    teacher_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-large')
    teacher_model.eval()

    # -----------------------------
    # Student Model (FLAN-T5 Base)
    # -----------------------------
    student_model = SmallT5(model_name='google/flan-t5-base')

    # -----------------------------
    # Distillation Loss Function
    # -----------------------------
    def distillation_loss(student_logits, teacher_logits, labels, temperature, alpha):
        # Soft loss using KL divergence
        soft_loss = F.kl_div(
            F.log_softmax(student_logits / temperature, dim=-1),
            F.softmax(teacher_logits / temperature, dim=-1),
            reduction='batchmean'
        ) * (temperature ** 2)

        # Hard loss using cross entropy
        hard_loss = F.cross_entropy(student_logits.view(-1, student_logits.size(-1)), labels.view(-1))

        return alpha * soft_loss + (1 - alpha) * hard_loss

    # -----------------------------
    # Optimizer and Hyperparameters
    # -----------------------------
    optimizer = torch.optim.Adam(student_model.parameters(), lr=0.0001)
    temperature = 5.0
    alpha = 0.8

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    teacher_model.to(device)
    student_model.to(device)

    # -----------------------------
    # Training Loop
    # -----------------------------
    num_epochs = 3

    for epoch in range(num_epochs):
        student_model.train()
        running_loss = 0.0
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Get teacher's predictions
            with torch.no_grad():
                teacher_outputs = teacher_model(input_ids, attention_mask=attention_mask)
                teacher_logits = teacher_outputs.logits

            # Get student's predictions
            student_outputs = student_model(input_ids, attention_mask=attention_mask)
            student_logits = student_outputs

            # Compute distillation loss
            loss = distillation_loss(student_logits, teacher_logits, labels, temperature, alpha)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}")

        # After training loop
        torch.save(student_model.state_dict(), './distilled_model.pth')  # Saving just the state_dict is preferred for flexibility
        print("Model saved ")

    print("Training successful")


# -----------------------------
# Safe Multiprocessing for Windows
# -----------------------------
if __name__ == '__main__':
    main()


README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

c:\Anjali\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\datasets--squad_v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Map:   0%|          | 0/1303 [00:00<?, ? examples/s]

AttributeError: 'list' object has no attribute 'to'

#Define Knowledge Distillation Loss

In [11]:
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, labels, alpha=0.5, temperature=2):
    # KL Divergence Loss
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=-1),
        F.softmax(teacher_logits / temperature, dim=-1),
        reduction="batchmean"
    ) * (temperature ** 2)
    
    # Cross-Entropy Loss
    hard_loss = F.cross_entropy(student_logits, labels)
    
    # Combined Loss
    return alpha * soft_loss + (1 - alpha) * hard_loss

#Train the Student Model

In [ ]:
from transformers import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

student_model.to(device)
optimizer = AdamW(student_model.parameters(), lr=5e-5)

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_dataloader = DataLoader(test_dataset, batch_size=8)

epochs = 3
for epoch in range(epochs):
    student_model.train()
    total_loss = 0
    for batch in tqdm(train_dataloader):
        optimizer.zero_grad()

        student_logits = student_model(input_ids=batch["student_input_ids"].to(device),
                                       attention_mask=batch["student_attention_mask"].to(device)).logits

        loss = distillation_loss(
            student_logits=student_logits,
            teacher_logits=batch["teacher_logits"].to(device),
            labels=batch["labels"].to(device)
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_dataloader)}")


#Evaluate the Student Model

In [ ]:
from datasets import load_metric

# Example: ROUGE metric
metric = load_metric("rouge")

student_model.eval()
for batch in val_dataloader:
    with torch.no_grad():
        outputs = student_model.generate(input_ids=batch["student_input_ids"].to(device),
                                         attention_mask=batch["student_attention_mask"].to(device))
    decoded_preds = student_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    decoded_labels = student_tokenizer.batch_decode(batch["labels"], skip_special_tokens=True)
    
    metric.add_batch(predictions=decoded_preds, references=decoded_labels)

result = metric.compute()
print(result)

#Save the Fine-Tuned Student Model

In [ ]:
student_model.save_pretrained("flan-t5-base-distilled")
student_tokenizer.save_pretrained("flan-t5-base-distilled")